# TP 1.1 — Your turn! Binary classification on circles

**IMT · Introduction to Machine Learning · Google Colab · Independent student exercise**

**The idea:** You have just seen the solution to TP 1 (two moons). Here you will apply the SAME steps to a new problem: **two concentric circles**. The changes are small on purpose—this is beginner practice.

**Learning goals:** build an MLP, choose the loss and optimizer, train, evaluate, and interpret the decision boundary.

### How to work
1. Open in Google Colab and choose **File → Save a copy in Drive**.
2. Run the provided cells in order. Complete the **four main ✍️ TODO cells** yourself.
3. Use the correction of TP 1 and your notes as a guide. Please work without generative AI assistance.
4. Answer the short written questions. The two **⭐ BONUS** exercises are optional.
5. Do not use **Runtime → Run all** before completing the TODO cells: unfinished cells intentionally raise an error.

**Suggested time:** 25–40 minutes. **No GPU needed.**

## Scenario: classifying two circular groups of points

The data contains **two concentric rings** of points. Each point has two coordinates and a class label (0 or 1).

Would a **straight line** be enough to separate an inner ring from an outer ring? We will compare logistic regression to a small neural network.

**Before you start:** Which model do you expect to perform better? Why?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

X, y = make_circles(n_samples=800, noise=0.16, factor=0.48, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
scaler = StandardScaler().fit(X_train)  # IMPORTANT: fit on training data only
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train[:, None], dtype=torch.float32)
xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test[:, None], dtype=torch.float32)
train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=12)
ax.set(title='Two circular groups (training data)', xlabel='Feature 1', ylabel='Feature 2')
plt.show()
print('Training samples:', len(X_train), '| Test samples:', len(X_test))

## 1. Start with a linear baseline (provided)

Just as in TP 1, fit **logistic regression** first. Its decision boundary is a straight line.

**Question:** Is its test accuracy close to 50%? Why is this dataset challenging for a straight-line classifier?

In [ ]:
baseline = LogisticRegression().fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
print(f'Linear baseline test accuracy: {baseline_accuracy:.3f}')

## 2. ✍️ TODO — define the neural network

Use `torch.nn.Sequential` again, but change the hidden-layer sizes from **16 to 12**:

- Input: **2** features
- Hidden layer: **12** neurons + ReLU
- Hidden layer: **12** neurons + ReLU
- Output: **1 logit** (no sigmoid at the end)

**Hint:** `nn.Linear(in_features, out_features)` and `nn.ReLU()`.

**Checkpoint:** `make_model()(xtr[:5]).shape` should be `(5, 1)`.

In [ ]:
def make_model():
    # TODO: return a torch.nn.Sequential network with 2 -> 12 -> 12 -> 1.
    # Hint: nn.Linear(in_features, out_features), nn.ReLU()
    raise NotImplementedError('Implement make_model()')

model = make_model()
assert model(xtr[:5]).shape == (5, 1)
print(model)

## 3. ✍️ TODO — loss and optimizer

Use the **same binary-classification loss** as TP 1: `nn.BCEWithLogitsLoss()`.
Use Adam with learning rate **0.01** and remember to provide `model.parameters()`.

**Question:** Why do we need a classification loss rather than simply comparing predicted and real coordinates?

In [ ]:
# TODO: assign `criterion` and `optimizer`.
raise NotImplementedError('Choose a classification loss and optimizer')

print(type(criterion).__name__, type(optimizer).__name__)

## 4. ✍️ TODO — implement the training loop

Fill in the same five steps as in TP 1:

1. Clear previous gradients using the optimizer.
2. Calculate `logits` from the model.
3. Compute `loss` using `criterion` and the true labels.
4. Backpropagate.
5. Update the parameters.

**Hints:** `optimizer.zero_grad()`, `model(xb)`, `criterion(...)`, `loss.backward()`, `optimizer.step()`.

The prepared code trains for **80 epochs** and plots the loss.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        # TODO: implement the five steps above.
        raise NotImplementedError('Implement the mini-batch training step')
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)

history = []
for epoch in range(80):
    history.append(train_one_epoch(model, train_loader, criterion, optimizer))

plt.plot(history)
plt.xlabel('Epoch'); plt.ylabel('Training BCE loss'); plt.title('Learning curve')
plt.show()

## 5. ✍️ TODO — probabilities and predictions

The model outputs **logits**. Use `torch.sigmoid` to convert them into probabilities, then choose class 1 when probability is **at least 0.5**.

**Hints:** `.numpy().ravel()` produces a one-dimensional NumPy array; `(probabilities >= 0.5).astype(int)` turns probabilities into class labels.

**Checkpoint:** the probabilities must be between 0 and 1. The resulting test accuracy should generally improve on logistic regression.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(xte)
    # TODO: compute `test_prob` and `test_pred` (integer array of 0s/1s).
    raise NotImplementedError('Apply sigmoid and threshold at 0.5')

assert test_prob.min() >= 0 and test_prob.max() <= 1
nn_accuracy = accuracy_score(y_test, test_pred)
print(f'Neural network test accuracy: {nn_accuracy:.3f} (linear: {baseline_accuracy:.3f})')
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap='Blues')
plt.show()

## 6. Visualize what the models learned (provided)

Run this cell without changing it. Compare the **straight-line boundary** of logistic regression with the **curved boundary** learned by the network.

**Think:** Why is a non-linear activation useful here?

In [ ]:
def plot_boundary(ax, predict_fn, title):
    grid_x, grid_y = np.meshgrid(
        np.linspace(X_test[:, 0].min()-0.6, X_test[:, 0].max()+0.6, 170),
        np.linspace(X_test[:, 1].min()-0.6, X_test[:, 1].max()+0.6, 170),
    )
    grid = np.c_[grid_x.ravel(), grid_y.ravel()]
    probs = predict_fn(grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, probs, levels=np.linspace(0,1,11), cmap='coolwarm', alpha=.6)
    ax.contour(grid_x, grid_y, probs, levels=[.5], colors='k', linewidths=1)
    ax.scatter(X_test[:,0], X_test[:,1], c=y_test, cmap='coolwarm', edgecolor='k', linewidth=.3, s=16)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
plot_boundary(axes[0], lambda z: baseline.predict_proba(z)[:,1], 'Logistic regression')
def predict_nn(z):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(z, dtype=torch.float32))).numpy().ravel()
plot_boundary(axes[1], predict_nn, 'Neural network')
plt.tight_layout(); plt.show()

## 7. Short written answers (required)

**Q1.** Which model had higher test accuracy? Write down the two numbers.

**My answer:** _Edit this markdown cell._

**Q2.** Why are the ReLU activations helpful for circular data?

**My answer:** _Edit this markdown cell._

**Q3.** Why should we evaluate on the test set, instead of only the training set?

**My answer:** _Edit this markdown cell._

---

**Optional bonus exercises below.** Complete these only after the main tasks work.

## ⭐ BONUS 1 — Changing the decision threshold

The main task uses a **0.5** probability threshold. Try **0.3, 0.5, and 0.7**.

For each threshold, compute the test accuracy and the number of **false positives** (a class-0 point predicted as class 1).

**Hints:** Use `accuracy_score(y_test, predictions)` and `np.sum((predictions == 1) & (y_test == 0))`.

**Bonus question:** Can reducing false positives be more important than maximizing overall accuracy? Give one example.

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    # BONUS TODO 1: Create predictions from test_prob with this threshold.
    # BONUS TODO 2: Calculate test accuracy and the number of false positives.
    # BONUS TODO 3: Print threshold, accuracy, and false-positive count.
    raise NotImplementedError('Bonus 1: compare the three thresholds')

## ⭐ BONUS 2 — How many hidden neurons do you need?

Our model uses **12 neurons** in each hidden layer. Try a new network using a variable number of neurons: **4, 12, or 32**.

Complete only the `candidate` model definition below. The rest of the code already trains and evaluates it for you.

**Hint:** Build `nn.Sequential` with `2 -> width -> width -> 1` and **ReLU between linear layers**.

**Bonus question:** Does the largest model always give the best test accuracy? Why might that not be guaranteed?

In [ ]:
def accuracy_with_width(width):
    torch.manual_seed(SEED)
    # BONUS TODO: Define a new model named `candidate` with 2 -> width -> width -> 1.
    # Hint: copy the structure from make_model(), replacing 12 with width.
    raise NotImplementedError('Bonus 2: define the candidate neural network')

    candidate_optimizer = torch.optim.Adam(candidate.parameters(), lr=0.01)
    for epoch in range(80):
        train_one_epoch(candidate, train_loader, criterion, candidate_optimizer)

    candidate.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(candidate(xte)).numpy().ravel()
    predictions = (probabilities >= 0.5).astype(int)
    return accuracy_score(y_test, predictions)

for width in [4, 12, 32]:
    score = accuracy_with_width(width)
    print(f'{width:2d} hidden neurons -> test accuracy: {score:.3f}')

---

### Before finishing

- [ ] Completed all **four main TODO cells** (model, loss/optimizer, training, predictions)
- [ ] Executed the notebook from top to bottom without errors
- [ ] Answered the three required discussion questions
- [ ] If attempted, included the **bonus code and explanation**
- [ ] Saved my own notebook copy

**Takeaway:** Non-linear neural networks can learn boundaries that simple linear classifiers cannot.